a temporary validation cell

In [ ]:
DELETE_TEST_ID = "delete-test-listing-001"

baseline_silver_ids = {
    row["listing_id"]
    for row in (
        spark.table(TARGET_TABLE)
        .select("listing_id")
        .collect()
    )
}

print("Baseline Silver IDs:", sorted(baseline_silver_ids))
print("Baseline Silver count:", len(baseline_silver_ids))

assert DELETE_TEST_ID not in baseline_silver_ids

In [ ]:
INSERT INTO shipp.listings (
    listing_id,
    donor_id,
    title,
    description,
    category,
    condition,
    location,
    latitude,
    longitude,
    available_from,
    available_until,
    status
)
VALUES (
    'delete-test-listing-001',
    'demo-donor-001',
    'DELETE CDC Test Listing',
    'Temporary listing created only to validate CDC delete semantics.',
    'FURNITURE',
    'GOOD',
    'Abu Dhabi',
    24.4976,
    54.4075,
    CURRENT_TIMESTAMP,
    CURRENT_TIMESTAMP + INTERVAL '7 days',
    'AVAILABLE'
);

In [ ]:
bronze_test_df = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("listing_id") == DELETE_TEST_ID)
    .select(
        "listing_id",
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
        "status",
        "updated_at",
    )
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
)

display(bronze_test_df)

In [ ]:
test_listing_count = (
    spark.table(TARGET_TABLE)
    .filter(F.col("listing_id") == DELETE_TEST_ID)
    .count()
)

print("Temporary listing in Silver:", test_listing_count)

assert test_listing_count == 1

print("PASS — Temporary listing reached Silver before delete.")

In [ ]:
bronze_df = spark.table(SOURCE_TABLE)

delete_history_df = (
    bronze_df
    .filter(F.col("listing_id") == DELETE_TEST_ID)
    .select(
        "listing_id",
        "_pg_change_type",
        "_pg_lsn",
        "_sort_by",
        "status",
        "updated_at",
    )
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
)

display(delete_history_df)

In [ ]:
latest_test_event = delete_history_df.first()

print("Latest change type:", latest_test_event["_pg_change_type"])
print("Latest LSN:", latest_test_event["_pg_lsn"])
print("Latest sort:", latest_test_event["_sort_by"])

In [ ]:
CURRENT_STATE_CHANGE_TYPES = {
    "insert",
    "update_postimage",
    "delete",
}

VALID_SILVER_SOURCE_CHANGE_TYPES = {
    "insert",
    "update_postimage",
}

In [ ]:
current_state_events_df = (
    bronze_df
    .filter(
        F.col("_pg_change_type").isin(
            list(CURRENT_STATE_CHANGE_TYPES)
        )
    )
)

print(
    "Current-state CDC rows:",
    current_state_events_df.count()
)

In [ ]:
latest_window = (
    Window
    .partitionBy(BUSINESS_KEY)
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
)

ranked_df = (
    current_state_events_df
    .withColumn(
        "_current_rn",
        F.row_number().over(latest_window)
    )
)

In [ ]:
latest_events_df = (
    ranked_df
    .filter(F.col("_current_rn") == 1)
)

deleted_latest_df = (
    latest_events_df
    .filter(F.col("_pg_change_type") == "delete")
)

current_rows_df = (
    latest_events_df
    .filter(F.col("_pg_change_type") != "delete")
)

print("Latest logical entities:", latest_events_df.count())
print("Latest deleted entities:", deleted_latest_df.count())
print("Current Silver entities:", current_rows_df.count())

In [ ]:
silver_df = (
    current_rows_df
    .select(
        F.col("listing_id"),
        F.col("donor_id"),
        F.col("title"),
        F.col("description"),
        F.col("category"),
        F.col("condition"),
        F.col("latitude"),
        F.col("longitude"),
        F.col("available_from"),
        F.col("available_until"),
        F.col("status"),
        F.col("updated_at"),
        F.col("_pg_lsn").alias("source_pg_lsn"),
        F.col("_sort_by").alias("source_sort_by"),
        F.col("_pg_change_type").alias("source_change_type"),
        F.current_timestamp().alias("silver_processed_at"),
    )
)

In [ ]:
invalid_source_change_types = (
    silver_df
    .filter(
        ~F.col("source_change_type").isin(
            list(VALID_SILVER_SOURCE_CHANGE_TYPES)
        )
    )
    .count()
)

In [ ]:
deleted_listing_in_silver = (
    spark.table(TARGET_TABLE)
    .filter(F.col("listing_id") == DELETE_TEST_ID)
    .count()
)

print(
    "Deleted test listing remaining in Silver:",
    deleted_listing_in_silver
)

assert deleted_listing_in_silver == 0, (
    "DELETE VALIDATION FAILED: "
    "deleted listing still exists in Silver."
)

print(
    "PASS — Deleted listing is absent from Silver."
)

In [ ]:
final_silver_ids = {
    row["listing_id"]
    for row in (
        spark.table(TARGET_TABLE)
        .select("listing_id")
        .collect()
    )
}

print("Baseline IDs:", sorted(baseline_silver_ids))
print("Final IDs:   ", sorted(final_silver_ids))

assert final_silver_ids == baseline_silver_ids, (
    "DELETE VALIDATION FAILED: "
    "unrelated Silver listings changed."
)

print(
    "PASS — Other Silver listings remain unchanged."
)

In [ ]:
expected_source_df = (
    current_rows_df
    .select(
        "listing_id",
        F.col("_pg_lsn").alias("expected_pg_lsn"),
        F.col("_sort_by").alias("expected_sort_by"),
        F.col("_pg_change_type").alias("expected_change_type"),
    )
)

In [ ]:
deleted_ids_df = (
    deleted_latest_df
    .select("listing_id")
    .distinct()
)

deleted_rows_still_in_silver = (
    spark.table(TARGET_TABLE)
    .join(
        deleted_ids_df,
        on="listing_id",
        how="inner",
    )
    .count()
)

print(
    "Deleted entities still present in Silver:",
    deleted_rows_still_in_silver
)

assert deleted_rows_still_in_silver == 0

print(
    "PASS — Latest DELETE events are excluded from Silver."
)